# Inverse Navier-Stokes: Inferring Re from Flow Data

**Time: ~45 minutes**

In notebook 07 we inferred a scalar ODE parameter (`k` in `u' = -ku`). Now we scale that idea to a **PDE inverse problem**: given sparse, noisy velocity measurements of a 2D flow, infer the **Reynolds number** while simultaneously reconstructing the full velocity and pressure fields.

This is the methodology behind the headline result in Raissi et al. (2019) — and notebook 08 identified it as PINNs' strongest use case.

### What's new here (beyond notebook 07)

- 2D Navier-Stokes equations (not a scalar ODE)
- Kovasznay flow — an exact NS solution we can validate against
- Pressure is **never observed** but recovered purely from physics
- Full NS residual: momentum (x, y) + continuity
- The streamfunction trick: eliminating continuity by construction

### Prerequisites

Notebooks 01-08, especially:
- 05 (PDE residuals with autograd)
- 07 (inverse PINNs with learnable parameters)

In [ ]:
import torch
import torch.nn as nn
import torch.autograd as autograd
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

---
## 1. The Kovasznay Flow — An Exact NS Benchmark

The 2D steady incompressible Navier-Stokes equations:

$$u \, u_x + v \, u_y = -p_x + \frac{1}{\text{Re}}(u_{xx} + u_{yy})$$
$$u \, v_x + v \, v_y = -p_y + \frac{1}{\text{Re}}(v_{xx} + v_{yy})$$
$$u_x + v_y = 0 \quad \text{(continuity)}$$

Kovasznay (1948) found an **exact solution** valid for any Re:

$$\lambda = \frac{\text{Re}}{2} - \sqrt{\frac{\text{Re}^2}{4} + 4\pi^2}$$

$$u(x,y) = 1 - e^{\lambda x} \cos(2\pi y)$$
$$v(x,y) = \frac{\lambda}{2\pi} e^{\lambda x} \sin(2\pi y)$$
$$p(x,y) = \frac{1}{2}(1 - e^{2\lambda x})$$

This is a perfect inverse problem benchmark: we can generate exact data, add noise, infer Re, and check our answer.

In [ ]:
# Domain and true Reynolds number
X_RANGE = (-0.5, 1.0)
Y_RANGE = (-0.5, 1.5)
RE_TRUE = 20.0

def kovasznay_lambda(re):
    """Eigenvalue for the Kovasznay solution."""
    return re / 2 - np.sqrt(re**2 / 4 + 4 * np.pi**2)

def exact_kovasznay(x, y, re):
    """Exact (u, v, p) for the Kovasznay flow."""
    lam = kovasznay_lambda(re)
    u = 1.0 - np.exp(lam * x) * np.cos(2 * np.pi * y)
    v = (lam / (2 * np.pi)) * np.exp(lam * x) * np.sin(2 * np.pi * y)
    p = 0.5 * (1.0 - np.exp(2 * lam * x))
    return u, v, p

# Visualise the true flow field
nx, ny = 80, 80
x_grid = np.linspace(*X_RANGE, nx)
y_grid = np.linspace(*Y_RANGE, ny)
X, Y = np.meshgrid(x_grid, y_grid)
U, V, P = exact_kovasznay(X, Y, RE_TRUE)
speed = np.sqrt(U**2 + V**2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].contourf(X, Y, U, 20, cmap='RdBu_r')
axes[0].set_title('u (x-velocity)'); plt.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(X, Y, V, 20, cmap='RdBu_r')
axes[1].set_title('v (y-velocity)'); plt.colorbar(im1, ax=axes[1])

im2 = axes[2].contourf(X, Y, P, 20, cmap='viridis')
axes[2].set_title('p (pressure)'); plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
plt.suptitle(f'Kovasznay Flow at Re = {RE_TRUE}', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"lambda = {kovasznay_lambda(RE_TRUE):.4f}")
print(f"The flow is periodic in y and decays exponentially in x.")

---
## 2. Generate Noisy Observations

In a real inverse problem, we'd have sensor measurements — scattered, noisy, and incomplete. We simulate this by:

1. Sampling random `(x, y)` locations in the domain
2. Evaluating the exact `(u, v)` at those points
3. Adding Gaussian noise

**Crucially, we never observe pressure.** The PINN will reconstruct it purely from the NS equations.

In [ ]:
N_OBS = 200   # number of observation points
NOISE = 0.01  # 1% noise (relative to signal std)

rng = np.random.default_rng(42)
x_obs = rng.uniform(*X_RANGE, N_OBS)
y_obs = rng.uniform(*Y_RANGE, N_OBS)

u_exact, v_exact, _ = exact_kovasznay(x_obs, y_obs, RE_TRUE)
u_obs = u_exact + NOISE * rng.standard_normal(N_OBS) * np.std(u_exact)
v_obs = v_exact + NOISE * rng.standard_normal(N_OBS) * np.std(v_exact)

# Convert to tensors
x_d = torch.tensor(x_obs, dtype=torch.float32).unsqueeze(1)
y_d = torch.tensor(y_obs, dtype=torch.float32).unsqueeze(1)
u_d = torch.tensor(u_obs, dtype=torch.float32).unsqueeze(1)
v_d = torch.tensor(v_obs, dtype=torch.float32).unsqueeze(1)

# Visualise observations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc0 = axes[0].scatter(x_obs, y_obs, c=u_obs, s=15, cmap='RdBu_r')
axes[0].set_title(f'Observed u ({N_OBS} points, {NOISE*100:.0f}% noise)')
plt.colorbar(sc0, ax=axes[0])

sc1 = axes[1].scatter(x_obs, y_obs, c=v_obs, s=15, cmap='RdBu_r')
axes[1].set_title(f'Observed v ({N_OBS} points, {NOISE*100:.0f}% noise)')
plt.colorbar(sc1, ax=axes[1])

for ax in axes:
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

print(f"We observe (u, v) at {N_OBS} scattered points.")
print(f"We do NOT observe pressure. The PINN must infer it from physics alone.")

---
## 3. The Inverse PINN Model

The model has two kinds of trainable parameters:

1. **Network weights** (~thousands) — learn the mapping `(x, y) -> (u, v, p)`
2. **`log_Re`** (1 scalar) — the unknown Reynolds number

We use `log_Re` (not `Re` directly) to ensure Re > 0, just like `log_k` in notebook 07.

In [ ]:
class InverseNSPINN(nn.Module):
    """PINN that outputs (u, v, p) and learns Re."""

    def __init__(self, hidden_layers=5, hidden_neurons=40, re_init=10.0):
        super().__init__()
        layers = [nn.Linear(2, hidden_neurons), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_neurons, hidden_neurons), nn.Tanh()]
        layers.append(nn.Linear(hidden_neurons, 3))  # outputs: u, v, p
        self.net = nn.Sequential(*layers)

        # Learnable parameter: Re in log-space
        self.log_re = nn.Parameter(torch.tensor(np.log(re_init)))

    @property
    def re(self):
        return torch.exp(self.log_re)

    def forward(self, x, y):
        out = self.net(torch.cat([x, y], dim=1))
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]  # u, v, p

RE_INIT = 10.0  # deliberately wrong — true is 20
model = InverseNSPINN(hidden_layers=5, hidden_neurons=40, re_init=RE_INIT)

n_params = sum(p.numel() for p in model.parameters())
print(f"Network parameters: {n_params - 1}")
print(f"Learnable physics parameter: log_Re (1 scalar)")
print(f"Initial Re guess: {model.re.item():.1f} (true: {RE_TRUE})")

---
## 4. The Two Loss Terms

This is the core of the inverse PINN methodology.

### Data loss
Match the observed velocity (NOT pressure — it was never measured):

$$\mathcal{L}_{\text{data}} = \frac{1}{N_{\text{obs}}} \sum_i \left[(u_{\text{pred}} - u_{\text{obs}})^2 + (v_{\text{pred}} - v_{\text{obs}})^2\right]$$

### Physics loss
The NS residual, using the **current estimate** of Re:

$$f_u = u \, u_x + v \, u_y + p_x - \frac{1}{\text{Re}}(u_{xx} + u_{yy})$$
$$f_v = u \, v_x + v \, v_y + p_y - \frac{1}{\text{Re}}(v_{xx} + v_{yy})$$
$$f_c = u_x + v_y$$

$$\mathcal{L}_{\text{physics}} = \frac{1}{N_{\text{coll}}} \sum_i \left[f_u^2 + f_v^2 + f_c^2\right]$$

The key insight: `Re` appears inside the physics loss. The optimizer adjusts `Re` to make the physics consistent with the data.

In [ ]:
def compute_ns_residual(model, x, y):
    """Compute the NS residual at collocation points.

    Returns (f_u, f_v, f_c) — momentum-x, momentum-y, continuity.
    """
    u, v, p = model(x, y)
    nu = 1.0 / model.re  # viscosity from learnable Re

    ones = torch.ones_like(u)

    # First derivatives
    u_x = autograd.grad(u, x, ones, create_graph=True)[0]
    u_y = autograd.grad(u, y, ones, create_graph=True)[0]
    v_x = autograd.grad(v, x, ones, create_graph=True)[0]
    v_y = autograd.grad(v, y, ones, create_graph=True)[0]
    p_x = autograd.grad(p, x, ones, create_graph=True)[0]
    p_y = autograd.grad(p, y, ones, create_graph=True)[0]

    # Second derivatives
    u_xx = autograd.grad(u_x, x, ones, create_graph=True)[0]
    u_yy = autograd.grad(u_y, y, ones, create_graph=True)[0]
    v_xx = autograd.grad(v_x, x, ones, create_graph=True)[0]
    v_yy = autograd.grad(v_y, y, ones, create_graph=True)[0]

    # NS residuals
    f_u = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    f_v = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)
    f_c = u_x + v_y  # continuity

    return f_u, f_v, f_c

print("NS residual function defined.")
print("Notice: nu = 1/Re uses the CURRENT estimate — this is what makes it inverse.")

---
## 5. Training

We minimise `L_data + L_physics` jointly over network weights AND `log_Re`.

In [ ]:
# Collocation points for the physics loss
N_COLL = 2000
x_coll = torch.rand(N_COLL, 1) * (X_RANGE[1] - X_RANGE[0]) + X_RANGE[0]
y_coll = torch.rand(N_COLL, 1) * (Y_RANGE[1] - Y_RANGE[0]) + Y_RANGE[0]
x_coll.requires_grad_(True)
y_coll.requires_grad_(True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

re_history = []
loss_data_hist = []
loss_phys_hist = []

EPOCHS = 15000
for epoch in range(EPOCHS):
    optimizer.zero_grad()

    # Data loss: match observed (u, v)
    u_pred, v_pred, _ = model(x_d, y_d)
    loss_data = torch.mean((u_pred - u_d)**2 + (v_pred - v_d)**2)

    # Physics loss: NS residual
    f_u, f_v, f_c = compute_ns_residual(model, x_coll, y_coll)
    loss_phys = torch.mean(f_u**2 + f_v**2 + f_c**2)

    loss = loss_data + loss_phys
    loss.backward()
    optimizer.step()

    re_history.append(model.re.item())
    loss_data_hist.append(loss_data.item())
    loss_phys_hist.append(loss_phys.item())

    if epoch % 3000 == 0:
        print(f"Epoch {epoch:5d} | Re = {model.re.item():.2f} | "
              f"data = {loss_data.item():.3e} | phys = {loss_phys.item():.3e}")

print(f"\nFinal Re = {model.re.item():.2f} (true: {RE_TRUE})")
print(f"Relative error: {abs(model.re.item() - RE_TRUE) / RE_TRUE * 100:.2f}%")

---
## 6. Re Convergence

Watch how the Reynolds number estimate evolves during training. It starts at 10 (wrong) and converges toward 20 (true).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Re convergence
axes[0].plot(re_history, color='#00205B', linewidth=1.5)
axes[0].axhline(y=RE_TRUE, color='red', linestyle='--', linewidth=2, label=f'True Re = {RE_TRUE}')
axes[0].axhline(y=RE_INIT, color='gray', linestyle=':', alpha=0.5, label=f'Initial guess = {RE_INIT}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Re')
axes[0].set_title('Reynolds Number Convergence')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss history
axes[1].semilogy(loss_data_hist, label='Data loss', color='#00205B', linewidth=1)
axes[1].semilogy(loss_phys_hist, label='Physics loss', color='#E85D04', linewidth=1)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss History')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. Evaluate: Velocity and Pressure Fields

Compare the PINN's predictions against the exact Kovasznay solution.

**The pressure plot is the most interesting**: the network has never seen pressure data, yet it reconstructs the pressure field purely from satisfying the NS equations.

In [ ]:
# Evaluate on a dense grid
x_eval = torch.tensor(X.flatten(), dtype=torch.float32).unsqueeze(1)
y_eval = torch.tensor(Y.flatten(), dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    u_pred, v_pred, p_pred = model(x_eval, y_eval)

u_pred_np = u_pred.numpy().reshape(ny, nx)
v_pred_np = v_pred.numpy().reshape(ny, nx)
p_pred_np = p_pred.numpy().reshape(ny, nx)

# Exact solution
U_exact, V_exact, P_exact = exact_kovasznay(X, Y, RE_TRUE)

# Pressure gauge invariance: mean-subtract both
p_pred_np = p_pred_np - p_pred_np.mean()
P_exact_centered = P_exact - P_exact.mean()

# Rel-L2 errors
vel_err = np.sqrt(np.sum((u_pred_np - U_exact)**2 + (v_pred_np - V_exact)**2))
vel_ref = np.sqrt(np.sum(U_exact**2 + V_exact**2))
p_err = np.sqrt(np.sum((p_pred_np - P_exact_centered)**2))
p_ref = np.sqrt(np.sum(P_exact_centered**2))

print(f"Velocity rel-L2: {vel_err / vel_ref:.4e}")
print(f"Pressure rel-L2: {p_err / p_ref:.4e}  (never observed during training!)")
print(f"Re error: {abs(model.re.item() - RE_TRUE) / RE_TRUE * 100:.2f}%")

In [ ]:
# Side-by-side comparison: exact vs PINN vs error
fields = [
    ('u (x-velocity)', U_exact, u_pred_np, 'RdBu_r'),
    ('v (y-velocity)', V_exact, v_pred_np, 'RdBu_r'),
    ('p (pressure, mean-subtracted)', P_exact_centered, p_pred_np, 'viridis'),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row, (name, exact, pred, cmap) in enumerate(fields):
    vmin = min(exact.min(), pred.min())
    vmax = max(exact.max(), pred.max())

    im0 = axes[row, 0].contourf(X, Y, exact, 20, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[row, 0].set_title(f'Exact {name}')
    plt.colorbar(im0, ax=axes[row, 0])

    im1 = axes[row, 1].contourf(X, Y, pred, 20, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[row, 1].set_title(f'PINN {name}')
    plt.colorbar(im1, ax=axes[row, 1])

    error = np.abs(pred - exact)
    im2 = axes[row, 2].contourf(X, Y, error, 20, cmap='Reds')
    axes[row, 2].set_title(f'|Error| (max={error.max():.3e})')
    plt.colorbar(im2, ax=axes[row, 2])

for ax in axes.flat:
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')

plt.suptitle(
    f'Inverse NS: Re_inferred = {model.re.item():.2f} (true = {RE_TRUE})',
    fontsize=14, fontweight='bold'
)
plt.tight_layout(); plt.show()

print("Bottom row: pressure was NEVER in the training data.")
print("The PINN reconstructed it purely from the NS equations.")

---
## 8. Pressure Gauge Invariance

Why did we mean-subtract the pressure before comparing?

The Navier-Stokes equations only involve **pressure gradients** ($p_x$ and $p_y$), never $p$ itself. So if $p(x,y)$ is a solution, then $p(x,y) + C$ is also a solution for any constant $C$.

This is called **gauge invariance**. The PINN can recover the pressure field up to an arbitrary constant — the physics doesn't pin down the absolute level. Mean-subtracting both fields removes this ambiguity for comparison.

---
## 9. The Streamfunction Trick

Our physics loss has three terms: momentum-x, momentum-y, and **continuity** ($u_x + v_y = 0$).

There's an elegant way to eliminate the continuity equation entirely. Instead of outputting $(u, v, p)$, output $(\psi, p)$ where $\psi$ is the **streamfunction**:

$$u = \frac{\partial \psi}{\partial y}, \quad v = -\frac{\partial \psi}{\partial x}$$

Then $u_x + v_y = \psi_{xy} - \psi_{xy} = 0$ **identically**. Continuity is satisfied by construction — no loss term needed.

This is exactly what `experiments/cylinder_wake/` does for the Raissi benchmark.

In [ ]:
class StreamfunctionPINN(nn.Module):
    """PINN using streamfunction: outputs (psi, p), derives (u, v)."""

    def __init__(self, hidden_layers=5, hidden_neurons=40, re_init=10.0):
        super().__init__()
        layers = [nn.Linear(2, hidden_neurons), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_neurons, hidden_neurons), nn.Tanh()]
        layers.append(nn.Linear(hidden_neurons, 2))  # outputs: psi, p
        self.net = nn.Sequential(*layers)
        self.log_re = nn.Parameter(torch.tensor(np.log(re_init)))

    @property
    def re(self):
        return torch.exp(self.log_re)

    def forward(self, x, y):
        out = self.net(torch.cat([x, y], dim=1))
        return out[:, 0:1], out[:, 1:2]  # psi, p

    def velocity(self, x, y):
        """Derive (u, v) from the streamfunction."""
        psi, p = self(x, y)
        ones = torch.ones_like(psi)
        u = autograd.grad(psi, y, ones, create_graph=True)[0]   # u = psi_y
        v = -autograd.grad(psi, x, ones, create_graph=True)[0]  # v = -psi_x
        return u, v, p

print("StreamfunctionPINN defined.")
print("u = psi_y, v = -psi_x => continuity (u_x + v_y = 0) is guaranteed.")
print("Physics loss only needs momentum equations — one fewer term to balance.")

---
## 10. What the Production Code Does Differently

Our notebook implementation is self-contained for learning. The production experiments in this repo add:

| Feature | Notebook | `experiments/navier_stokes_inverse/` | `experiments/cylinder_wake/` |
|---------|----------|-------------------------------------|-----------------------------|
| Data source | Synthetic (exact) | Synthetic (exact Kovasznay) | Real DNS data (1M points) |
| Network | 5×40, raw PyTorch | 5×64, `pinn.PINN` | 8×64, streamfunction |
| Training | Adam only | Adam + optional L-BFGS | Adam + L-BFGS (two-stage) |
| Learnable params | `log_Re` (1) | `log_Re` (1) | `lambda_1`, `lambda_2` (2) |
| Logging | print | loguru + file sinks | loguru + file sinks |
| Checkpoints | none | Self-describing `.pt` | Self-describing `.pt` |

The two-stage training (Adam then L-BFGS) typically improves parameter inference by 1-2 orders of magnitude. See `docs/dev-log.md` Phase 10 for details.

### Try them:

```bash
# Kovasznay inverse — fully self-contained
uv run train-ns-inverse train -e 30000 --re-init 10
uv run train-ns-inverse predict

# Cylinder wake — requires .mat data file
uv run train-cylinder train -e 5000 --lbfgs-epochs 2000
uv run train-cylinder predict
```

---
## Key Takeaways

1. **Inverse PINNs scale to PDEs.** The same `nn.Parameter` trick from notebook 07 works for the full Navier-Stokes equations — Re is just one more learnable scalar.

2. **Data loss + physics loss = parameter inference.** The data anchors the solution, and the physics loss forces the parameter to be consistent. Neither loss alone is sufficient.

3. **Hidden fields can be reconstructed.** Pressure was never observed, yet the PINN recovers it. This is genuinely useful — in real experiments, pressure sensors are often impractical while velocity measurements (e.g., PIV) are available.

4. **Gauge invariance matters.** Always mean-subtract pressure before comparing — the NS equations only constrain gradients, not the absolute level.

5. **The streamfunction trick** eliminates continuity by construction. Use it when incompressibility is a hard constraint.

## Exercises

1. **More noise**: Increase to 5% or 10%. How robust is the Re estimate? At what noise level does it break?

2. **Fewer observations**: Try 50, 20, 10 points. When does the problem become ill-posed?

3. **Wrong initial guess**: Start Re at 100 instead of 10. Does it still converge?

4. **Streamfunction version**: Train `StreamfunctionPINN` on the same problem. Compare convergence speed and final accuracy — does removing the continuity loss help?

5. **Two-parameter inference**: Modify the NS equations to use `lambda_1 * (u*u_x + v*u_y)` and `lambda_2 * (u_xx + u_yy)`, with both as learnable parameters. Can you recover both simultaneously? (This is what the cylinder wake experiment does.)

6. **Data weighting**: Try `loss = 10*loss_data + loss_phys`. Does stronger data weighting speed up Re convergence?